# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarKasiba/ML-Pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


* **One row means:** One specific content page (`content_id`) associated with a specific client (`client_id`) on a specific calendar day (`report_date`), tracking its daily search performance.
* **Time window:** Developed and tested on the mid-panel month **`month=2026-03`** (March 2026), reserving the final sample month (`2026-06`) as the sealed evaluation window.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Initialize DuckDB and authenticate with Hugging Face token
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# Quick test query using correct warehouse schema column names
query_sample = """
SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5;
"""
display(con.execute(query_sample).fetchdf())

,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


* **Feature:**
  - `lag_impressions_7d`: Sum of impressions over the preceding 7 days.
  - `lag_clicks_7d`: Sum of clicks over the preceding 7 days.
  - `avg_position_7d`: Average SERP ranking position over the preceding 7 days.
* **Label:**
  - `decay_flag`: Binary indicator (1 if traffic/impressions dropped by more than 25% compared to the prior window, signaling a content refresh opportunity; 0 otherwise).
* **Context:**
  - `client_id`, `content_id`, `report_date`.
* **Excluded:**
  - Raw page URLs and specific brand keywords. **Why:** Excluded to prevent high-cardinality overfitting, ensure privacy compliance, and focus the model strictly on behavioral performance metrics rather than brittle string matches.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect table schema to ensure all expected fields exist
schema_query = """
DESCRIBE SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet');
"""
display(con.execute(schema_query).fetchdf())

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

* **Grain Claim:** Each combination of `client_id`, `content_id`, and `report_date` is unique (no duplicate primary key rows per day).
* **Counts & Span Claim:** The dataset covers all days in March 2026 with consistent row counts.
* **Availability Claim:** Valid performance metrics exist when filtering with explicit boolean checks (`IS TRUE`).

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: Grain uniqueness check (should return 0 rows if grain holds)
grain_check = """
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS cnt
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5;
"""
print("--- Grain Check Duplicates ---")
display(con.execute(grain_check).fetchdf())

# Query 2: Row counts, date span, and missing value check
stats_query = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS active_days,
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("--- Counts and Span Statistics ---")
display(con.execute(stats_query).fetchdf())

# Query 3: Availability check using IS TRUE
availability_query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN (gsc_impressions >= 0) IS TRUE THEN 1 ELSE 0 END) as valid_impression_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet');
"""
print("--- Availability Check (`IS TRUE`) ---")
display(con.execute(availability_query).fetchdf())


--- Grain Check Duplicates ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,cnt


--- Counts and Span Statistics ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,active_days,total_rows,missing_impressions
0,2026-03-01,2026-03-31,31,9841378,0.0


--- Availability Check (`IS TRUE`) ---


,total_rows,valid_impression_rows
0,9841378,9841378.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **What this data can never tell you:**
  - **Qualitative intent & seasonal context:** It cannot distinguish whether a drop in traffic is caused by true content decay/obsolescence or external seasonal search shifts (e.g., holiday dips).
  - **Unbalanced client onboarding history:** Different clients enter the warehouse at different times, meaning older domains have deep longitudinal trends while newly onboarded clients suffer from sparse historical windows, requiring careful masking or cohort normalization.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Explore client activity span variations to verify unbalanced history limits
client_history_query = """
SELECT
    client_hash_id,
    MIN(report_date) AS first_observed,
    MAX(report_date) AS last_observed,
    COUNT(DISTINCT report_date) AS days_recorded
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY 1
ORDER BY days_recorded ASC
LIMIT 5;
"""
print("--- Client History Depth Variation Check ---")
display(con.execute(client_history_query).fetchdf())

--- Client History Depth Variation Check ---


,client_hash_id,first_observed,last_observed,days_recorded
0,client_e00b29e582949543,2026-03-23,2026-03-31,9
1,client_810019792c9b8efc,2026-03-20,2026-03-31,12
2,client_f6f0cdf26d03d7bd,2026-03-19,2026-03-31,13
3,client_86ebc2f12c01f586,2026-03-03,2026-03-31,29
4,client_8ae2bfb5aa1ffa1e,2026-03-01,2026-03-31,31


## Self-check

Before you submit, confirm each line honestly:

- [yes ] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.